# Colombia Open — TRM (Tasa de Cambio Representativa del Mercado)

Este notebook muestra cómo cargar el dataset **`macroeconomia_trm`** (TRM diaria, COP por USD, 1991 a la fecha) con `colombia-open` — la fuente más simple del catálogo: sin dimensión territorial, sin paginación, sin agregación.

> **Nota (2026-10):** `colombia-open` está publicado en [PyPI](https://pypi.org/project/colombia-open/) — `pip install colombia-open` descarga y cachea el bundle de datos publicado en GitHub Releases automáticamente. Este dataset está disponible desde el bundle `v0.11.0`.

## Setup

In [ ]:
!pip install -q colombia-open

## Cargar el dataset

In [ ]:
import polars as pl
from colombia_open import ColombiaOpen

hub = ColombiaOpen()
trm = hub.load_polars("macroeconomia_trm")

print(trm.shape)
trm.sort("fecha", descending=True).head(10)

## Esquema

| Columna | Tipo | Descripción |
|---|---|---|
| `fecha` | date | Inicio de la vigencia — clave primaria, única. |
| `vigente_hasta` | date | Fin de la vigencia (cubre fines de semana/festivos cuando aplica). |
| `trm` | float | Pesos colombianos (COP) por dólar (USD). Siempre > 0. |

Esquema completo: [`docs/datasets/macroeconomia_trm.md`](../../docs/datasets/macroeconomia_trm.md).

## Serie de tiempo: promedio mensual

In [ ]:
(
    trm.with_columns(pl.col("fecha").dt.strftime("%Y-%m").alias("mes"))
    .group_by("mes")
    .agg(pl.col("trm").mean().round(2).alias("trm_promedio"))
    .sort("mes", descending=True)
    .head(12)
)

## Máximo y mínimo histórico

In [ ]:
print("máximo:", trm.sort("trm", descending=True).head(1).to_dicts())
print("mínimo:", trm.sort("trm").head(1).to_dicts())

## Expandir a un valor por día calendario

In [ ]:
# Si necesitas un valor para cada día (incluidos los que la fuente no
# publicó por separado, ej. fines de semana), expande el rango de vigencia.
trm.select(pl.date_ranges("fecha", "vigente_hasta").alias("dia"), "trm").explode("dia").tail(10)

## Limitaciones conocidas

- **No es "un valor por día calendario" por defecto** — es "un valor por publicación" (ver celda de expansión arriba si lo necesitas día a día).
- **Serie nacional, sin desagregación territorial** — no tiene `cod_mpio`/`cod_dpto`.
- **El día en curso puede no estar publicado todavía** si cargas muy temprano.

Documentación completa: [`docs/datasets/macroeconomia_trm.md`](../../docs/datasets/macroeconomia_trm.md).